### teste

In [ ]:
from retail_analytics.config import DATA_FILE_PATH, logger
from retail_analytics.data_loader import DataLoader
from retail_analytics.metrics import RetailMetrics

In [ ]:
def load_data():
    loader = DataLoader(DATA_FILE_PATH)
    return loader.load_and_clean()


In [ ]:
try:
    df = load_data()
    metrics = RetailMetrics(df)
except Exception as e:
    print(f"Erro ao carregar os dados: {e}")
    #st.stop()

In [ ]:
# Verificar as novas dimensões e primeiras linhas
print(df.shape)
print(df.head())

In [ ]:
# 1. Converter a coluna de data para datetime
df["DATA"] = pd.to_datetime(df["DATA"], format="%d/%m/%Y")

# 2. Caso colunas numéricas tenham vindo como texto por causa de vírgula decimal (ex: "10,50")
# df['VALOR'] = df['VALOR'].str.replace(',', '.').astype(float)

# 3. Visão geral da base
df.info()

1. Preparação e Limpeza dos Dados
Antes das métricas, ajustamos tipos de dados e criamos colunas derivadas úteis para agregação:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configuração visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# 1. Carregamento correto
df = pd.read_csv("Base Varejo.csv", sep=";")

# 2. Conversão da Data e criação de variáveis temporais
df["DATA"] = pd.to_datetime(df["DATA"], format="%d/%m/%Y")
df["ANO_MES"] = df["DATA"].dt.to_period("M")
df["DIA_SEMANA"] = df["DATA"].dt.day_name()
df["MES"] = df["DATA"].dt.month

# 3. Tratamento de números com vírgula (se aplicável)
colunas_numericas = df.select_dtypes(include="object").columns
for col in colunas_numericas:
  if df[col].str.contains(r"^\d+,\d+$", na=False).any():
    df[col] = df[col].str.replace(",", ".").astype(float)

# Verificação geral de valores nulos e tipos
print(df.info())

2. Visão Geral do Negócio (KPIs Macro)
Métricas essenciais para entender a escala da base:

In [ ]:
# Identifique a coluna de valor total (ex: 'VALOR_TOTAL', 'VL_TOTAL' ou QTD * PRECO)
col_valor = "VALOR_TOTAL"  # ajuste para o nome exato da sua coluna de faturamento

total_faturamento = df[col_valor].sum() if col_valor in df.columns else None
total_cupons = df["CO_ID"].nunique()
total_clientes = df["CL_ID"].nunique()
total_itens_vendidos = len(df)

print(f"--- RESUMO EXECUTIVO ---")
if total_faturamento:
  print(f"Faturamento Total: R$ {total_faturamento:,.2f}")
  print(
      f"Ticket Médio por Cupom: R$ {df.groupby('CO_ID')[col_valor].sum().mean():.2f}"
  )
print(f"Total de Compras (Cupons): {total_cupons:,}")
print(f"Total de Clientes Únicos: {total_clientes:,}")
print(
    f"Média de Itens por Cupom: {df.groupby('CO_ID')['PR_ID'].count().mean():.1f} itens"
)

3. Análise de Produtos e Categorias
Responde a: Quais categorias e produtos movem o negócio?

In [ ]:
# 1. Faturamento / Volume por Categoria
if col_valor in df.columns:
  cat_resumo = (
      df.groupby("PR_CAT")
      .agg(
          faturamento=(col_valor, "sum"),
          volume_itens=("PR_ID", "count"),
          produtos_distintos=("PR_ID", "nunique"),
      )
      .sort_values(by="faturamento", ascending=False)
  )
else:
  cat_resumo = (
      df["PR_CAT"].value_counts().to_frame(name="volume_itens_vendidos")
  )

print(cat_resumo)

# Gráfico de barras das Categorias
sns.countplot(
    data=df,
    y="PR_CAT",
    order=df["PR_CAT"].value_counts().index,
    palette="Blues_r",
)
plt.title("Volume de Itens Vendidos por Categoria")
plt.xlabel("Qtd. de Linhas Vendidas")
plt.ylabel("Categoria")
plt.show()

# 2. Top 10 Produtos Mais Vendidos
top_produtos = df["PR_NOME"].value_counts().head(10)
print("\n--- TOP 10 PRODUTOS MAIS VENDIDOS ---")
print(top_produtos)

4. Análise de Perfil e Segmentação de Clientes
Responde a: Quem compra com a gente e qual o perfil de maior valor?

In [ ]:
# 1. Perfil demográfico (Gênero e Segmento)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df.drop_duplicates(subset=["CL_ID"]), x="CL_GENERO", ax=axes[0], palette="pastel")
axes[0].set_title("Distribuição de Clientes por Gênero")

sns.countplot(data=df.drop_duplicates(subset=["CL_ID"]), x="CL_SEG", order=sorted(df["CL_SEG"].dropna().unique()), ax=axes[1], palette="Set2")
axes[1].set_title("Distribuição de Clientes por Segmento")
plt.show()

# 2. Frequência de compra por cliente (quantas compras cada um fez)
compras_por_cliente = df.groupby("CL_ID")["CO_ID"].nunique()
print("--- FREQUÊNCIA DE COMPRA ---")
print(f"Média de compras por cliente: {compras_por_cliente.mean():.2f}")
print(f"Mediana de compras por cliente: {compras_por_cliente.median():.0f}")
print(f"Máximo de compras por um único cliente: {compras_por_cliente.max()}")

5. Análise Temporal e Sazonalidade
Responde a: Como as vendas se comportam ao longo do tempo e nos dias da semana?

In [ ]:
# 1. Evolução Mensal de Vendas
vendas_mensais = df.groupby("ANO_MES")["CO_ID"].nunique()
vendas_mensais.plot(kind="line", marker="o", color="navy")
plt.title("Evolução Mensal do Número de Transações (Cupons)")
plt.ylabel("Qtd. Cupons")
plt.xlabel("Mês")
plt.xticks(rotation=45)
plt.show()

# 2. Concentração por Dia da Semana
ordem_dias = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
df["DIA_SEMANA"] = pd.Categorical(
    df["DIA_SEMANA"], categories=ordem_dias, ordered=True
)

cupons_dia_semana = df.groupby("DIA_SEMANA", observed=True)["CO_ID"].nunique()
cupons_dia_semana.plot(kind="bar", color="steelblue")
plt.title("Volume de Transações por Dia da Semana")
plt.ylabel("Qtd. Cupons")
plt.xlabel("Dia")
plt.show()